# Part 11
## 11.1 - 11.4: Quantitative Evaluation

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_json('/media/trong-thanh/Data/University/Fourth Year/NLP/experiments/ex1/data.json', lines=True)
texts = df['text'].fillna('').tolist()

vectorizer = CountVectorizer(lowercase=True, stop_words='english', token_pattern=r'(?u)\b\w+\b')
X = vectorizer.fit_transform(texts)
tfidf_trans = TfidfTransformer()
X_tfidf = tfidf_trans.fit_transform(X)

Loaded data.


In [2]:
eval_set = {
    "medical image classification": [18971, 12658],
    "deep learning healthcare": [6123, 8270],
    "natural language processing": [5912, 23722],
    "transformer language model": [18339],
    "heart attack treatment": [799, 1422]
}

def evaluate(queries_dict, k=5):
    rows = []
    for q, rel in queries_dict.items():
        q_vec = tfidf_trans.transform(vectorizer.transform([q]))
        sims = cosine_similarity(q_vec, X_tfidf).flatten()
        ret = [int(x) for x in np.argsort(sims)[::-1][:k]]
        
        hits = [d for d in ret if d in rel]
        p_k = len(hits) / k
        r_k = len(hits) / len(rel)
        
        rr = 0.0
        for rank, d in enumerate(ret, 1):
            if d in rel:
                rr = 1.0 / rank
                break
                
        rows.append({
            'query': q,
            'p_at_5': round(p_k, 4),
            'r_at_5': round(r_k, 4),
            'mrr': round(rr, 4),
            'retrieved': ret,
            'relevant': rel
        })
    return pd.DataFrame(rows)

results = evaluate(eval_set)
display(results)
print(f"Mean P@5: {results['p_at_5'].mean():.4f}")
print(f"Mean R@5: {results['r_at_5'].mean():.4f}")
print(f"MRR: {results['mrr'].mean():.4f}")


query,p_at_5,r_at_5,mrr,retrieved,relevant
medical image classification,0.4,1.0,1.0,"[18971, 8527, 19908, 12658, 15682]","[18971, 12658]"
deep learning healthcare,0.2,0.5,0.5,"[7564, 6123, 11979, 11119, 9252]","[6123, 8270]"
natural language processing,0.0,0.0,0.0,"[25428, 8705, 24482, 4075, 701]","[5912, 23722]"
transformer language model,0.0,0.0,0.0,"[27936, 25428, 24482, 4075, 701]",[18339]
heart attack treatment,0.4,1.0,1.0,"[799, 1422, 11245, 22876, 16626]","[799, 1422]"


Mean P@5: 0.2000
Mean R@5: 0.5000
MRR: 0.5000


In [3]:
results.to_csv('results.csv', index=False)

## 12. Part I — Error Analysis

Trong năm câu truy vấn thử nghiệm, medical image classification và heart attack treatment đạt kết quả cao nhất với MRR và R@5 đều bằng 1.0. Các tài liệu đứng đầu như số 18971 và 799 đều chứa đúng các từ khóa quan trọng mang giá trị IDF lớn như classification hay cụm từ heart attack. Sự trùng khớp từ vựng trực tiếp giúp hệ thống xếp đúng toàn bộ tài liệu y khoa cần tìm vào nhóm dẫn đầu, dù đôi khi vẫn bị lọt vào vài văn bản nhiễu do trùng lặp đơn lẻ từng từ.


In [5]:
good_queries = ["medical image classification", "heart attack treatment"]
for q in good_queries:
    q_vec = tfidf_trans.transform(vectorizer.transform([q]))
    sims = cosine_similarity(q_vec, X_tfidf).flatten()
    top_idx = int(np.argsort(sims)[::-1][0])
    print(f"Query '{q}' -> Top 1 Doc {top_idx} | Similarity {sims[top_idx]:.4f}")
    print(f"Trich doan {texts[top_idx].replace(chr(10), ' ')[:100]}...\n")


Query 'medical image classification' -> Top 1 Doc 18971 | Similarity 0.4237
Trich doan The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commission...

Query 'heart attack treatment' -> Top 1 Doc 799 | Similarity 0.3962
Trich doan Can You Have A Heart Attack With A Pacemaker? A pacemaker is not a cure. It will not prevent or stop...



Ngược lại, hai câu truy vấn natural language processing và deep learning healthcare thất bại nặng nề do hạn chế của mô hình túi từ. Khi tìm kiếm về xử lý ngôn ngữ tự nhiên, hệ thống bị điểm 0 hoàn toàn vì bẻ gãy cụm từ thành các từ rời rạc, dẫn đến việc xếp đầu các bài viết về đổi ngôn ngữ app Facebook và an toàn chế biến thực phẩm. Với câu truy vấn về học sâu y tế, một bảng chia thì động từ lặp đi lặp lại chữ learning đã leo lên vị trí số một do tần suất TF tăng tuyến tính không bão hòa, đẩy lùi các bài báo khoa học thực sự.


In [6]:
bad_queries = ["natural language processing", "deep learning healthcare"]
for q in bad_queries:
    q_vec = tfidf_trans.transform(vectorizer.transform([q]))
    sims = cosine_similarity(q_vec, X_tfidf).flatten()
    top_idx = int(np.argsort(sims)[::-1][0])
    print(f"Query '{q}' -> Top 1 Doc {top_idx} | Similarity {sims[top_idx]:.4f}")
    print(f"Trich doan {texts[top_idx].replace(chr(10), ' ')[:100]}...\n")


Query 'natural language processing' -> Top 1 Doc 25428 | Similarity 0.3893
Trich doan Note: If you're on an iPhone, you cannot change the language of Facebook through the mobile app. Ins...

Query 'deep learning healthcare' -> Top 1 Doc 7564 | Similarity 0.3094
Trich doan would I have learned, learnt? would you have learned, learnt? would he have learned, learnt? would w...



Trường hợp thất bại rõ nhất là truy vấn heart attack treatment đối chiếu với bài viết số 8343 về liệu pháp nhồi máu cơ tim myocardial infarction therapy. Dù hai khái niệm hoàn toàn đồng nghĩa trong y khoa, điểm tương đồng cosine vẫn bằng đúng 0.0000 vì chúng không chung bất kỳ từ vựng nào. Bản chất trực giao của TF-IDF khiến hệ thống hoàn toàn bất lực trước hiện tượng từ đồng nghĩa và từ đa nghĩa. Điều này đòi hỏi chúng ta phải chuyển dịch sang các không gian biểu diễn ngữ nghĩa dày đặc như Word2Vec hay BERT để nắm bắt mối quan hệ ngữ cảnh thực sự.


In [7]:
doc_fail = next(i for i, t in enumerate(texts) if 'myocardial infarction' in t.lower() and 'therapy' in t.lower() and 'heart' not in t.lower())

q_fail = 'heart attack treatment'
q_fail_vec = tfidf_trans.transform(vectorizer.transform([q_fail]))
sim_fail = cosine_similarity(q_fail_vec, X_tfidf[doc_fail:doc_fail+1])[0, 0]

print(f"Doc ID {doc_fail}")
print(f"Trich doan {texts[doc_fail][:100]}...")
print(f"Similarity voi '{q_fail}' la {sim_fail:.4f}")


Doc ID 8343
Trich doan Background: Statin therapy is associated with improved survival in patients at high risk for cardiov...
Similarity voi 'heart attack treatment' la 0.0000
